In [1]:
import gym as gym
from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import (notebook_login,)
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import DQN

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
env = gym.make("LunarLander-v2")

observation, info = env.reset()

for _ in range(20):

    action = env.action_space.sample()
    print(f"Action {action} has been taken")

    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        print("Reset")
        observation, info = env.reset()

env.close()


Action 2 has been taken
Action 2 has been taken
Action 2 has been taken
Action 0 has been taken
Action 0 has been taken
Action 2 has been taken
Action 0 has been taken
Action 2 has been taken
Action 2 has been taken
Action 3 has been taken
Action 0 has been taken
Action 3 has been taken
Action 1 has been taken
Action 2 has been taken
Action 1 has been taken
Action 0 has been taken
Action 1 has been taken
Action 3 has been taken
Action 1 has been taken
Action 2 has been taken


/mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/GOD1/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [3]:
env = gym.make("LunarLander-v2")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [-1.3422542  -0.1992913  -2.3930085  -3.3631864  -2.835068   -4.932717
  0.796911    0.81001663]


In [4]:
print("_____ACTION SPACE______")
print(f"Action space shape {env.action_space.n}")
print(f"Action space sample {env.action_space.sample}")

_____ACTION SPACE______
Action space shape 4
Action space sample <bound method Discrete.sample of Discrete(4)>


In [ ]:
env = make_vec_env("LunarLander-v2", n_envs= 16)

model = PPO('MlpPolicy', env, verbose  = 1)

model.learn(total_timesteps = int(2e5))

In [7]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose = 1
)

model.learn(total_timesteps = 1000000)
model_name = "ppo-LunarLander-v2"
model.save(model_name)

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.3     |
|    ep_rew_mean     | -195     |
| time/              |          |
|    fps             | 1905     |
|    iterations      | 1        |
|    time_elapsed    | 8        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 87.4         |
|    ep_rew_mean          | -136         |
| time/                   |              |
|    fps                  | 1083         |
|    iterations           | 2            |
|    time_elapsed         | 30           |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0070491685 |
|    clip_fraction        | 0.0626       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -0.000535   

In [ ]:
import os
eval_env = Monitor(gym.make("LunarLander-v2"))

model_path = #specify path

if os.path.exists(model_path):
    print("yes")
else:
    print("no")

model = PPO.load(model_path , env = eval_env)

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes = 10, deterministic = True)
print(f"Mean Reward:{mean_r:.2f} +/- Std Reward:{std_r}")


In [ ]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
from stable_baselines3.common.env_util import make_vec_env

from huggingface_sb3 import package_to_hub

def _vec_video_recorder__getattr(self, name):
    if "video_recorder" == name:
        return self
    if "path" == name:
        return self.video_path
    if _orig_vec_video_recorder_getattr is None:
        raise AttributeError(name)
    return _orig_vec_video_recorder_getattr(self, name)

_orig_vec_video_recorder_getattr = getattr(VecVideoRecorder, "__getattr__", None)
setattr(VecVideoRecorder, "__getattr__", _vec_video_recorder__getattr)

repo_id = #"ypur repo id"

env_id = "LunarLander-v2"

eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode = "rgb_array")])

model_arch = "PPO"

video_folder = "logs/videos/"

video_length = 200

eval_env = VecVideoRecorder(
    eval_env,
    video_folder,
    record_video_trigger = lambda x: x == 0,
    video_length = video_length,
    name_prefix = f"ppo-{env_id}"
)

commit_msg = "commited"

package_to_hub(
    model = model,
    model_name = "RL-LunarLander-v2",
    model_architecture = model_arch,
    env_id = env_id,
    eval_env = eval_env,
    repo_id = repo_id,
    commit_message = commit_msg
)

In [ ]:
from huggingface_sb3 import load_from_hub

repo = #"your repo id"

file1 = "RL-LunarLander-v3.zip"

custom_objects = {
    "learning_rate": 0.0,
    "lr_schedule": lambda: 0.0,
    "clip_range": lambda: 0.0
}

checkpoint = load_from_hub(repo, file1)
model = PPO.load(checkpoint, custom_objects = custom_objects, print_systeminfo = True)

In [18]:
ENV = Monitor(gym.make("LunarLander-v3"))
reward, std_reward = evaluate_policy(model, ENV, n_eval_episodes = 10, deterministic = True)
print(f"Mean Reward:{reward:.2f} | Std Reward:{std_reward}")


Mean Reward:253.41 | Std Reward:43.81981122425167
